# 03 - From a probability to a decision

A classifier emits `p_bad`. That is not a decision. This notebook is about the
layer that turns the number into an action, which is where most of the
interesting engineering lives.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np, warnings
warnings.filterwarnings('ignore')
DATA = '../data300k'      # the working set
DATA100K = '../data'      # the calibrated baseline the pitch quotes

In [2]:
from core.feature_store import FeatureStore
from core.truth import TruthVault
from core.model import Adjudicator
from core.policy import PolicyConfig, decide, ev_release, Action
from core.backtest import run
from core.metrics import grade, StepUpModel, frontier, ev_release_ceiling

store, vault = FeatureStore.load(DATA), TruthVault(DATA)
model = Adjudicator().fit(store, vault)
cfg, su = PolicyConfig(), StepUpModel()
print(model.card.to_json())

{
  "blocks": [
    "local",
    "network"
  ],
  "features": [
    "f_device_is_new",
    "f_device_account_fanout",
    "f_address_mismatch",
    "f_orders_last_24h",
    "f_amount_z",
    "f_pincode_rto_propensity",
    "f_is_night",
    "f_thin_file_flag",
    "f_disposable_email",
    "f_international",
    "f_merchant_prior_rto",
    "f_is_cod",
    "risk_score",
    "amount",
    "network_orders_prior",
    "network_merchants_prior",
    "network_tenure_days",
    "network_clean_rate",
    "network_disputes_prior",
    "network_rto_prior",
    "network_device_fanout",
    "network_instrument_merchants"
  ],
  "n_fit": 5192,
  "n_calib": 1298,
  "base_rate_fit": 0.2553929121725732,
  "learner": "lightgbm",
  "seed": 42,
  "feature_hash": "1805b25305fb2171",
  "source": "../data300k\\appeal_queue.csv"
}


## Calibration is not optional here

The policy spends `p_bad` as a price. If the model says 0.2 and the true rate
is 0.4, the policy is not slightly wrong -- it is systematically buying bad
orders. Isotonic regression fitted on the last 20% of the train window fixes
this, and the reliability curve is how you check.

In [3]:
ho = store.split('holdout')
y  = vault.labels(store.payment_ids(ho))
p  = model.predict(store, ho)
raw = model.raw_uncalibrated(store, ho)

edges = np.linspace(0, 1, 11)
print(f'{"bucket":<14}{"n":>6}{"predicted":>12}{"observed":>11}{"error":>9}')
print('-'*55)
for b in range(10):
    m_ = (np.digitize(p, edges) - 1 == b)
    if m_.sum() < 5: continue
    print(f'{edges[b]:.1f}-{edges[b+1]:.1f}      {m_.sum():>6}{p[m_].mean():>12.3f}{y[m_].mean():>11.3f}{p[m_].mean()-y[m_].mean():>+9.3f}')
print(f'\ncalibration error: raw {np.abs(raw.mean()-y.mean()):.4f} -> isotonic {np.abs(p.mean()-y.mean()):.4f}')

bucket             n   predicted   observed    error
-------------------------------------------------------
0.0-0.1        1125       0.020      0.036   -0.015
0.1-0.2          30       0.138      0.200   -0.062
0.2-0.3         333       0.211      0.240   -0.029
0.3-0.4          94       0.364      0.383   -0.019
0.5-0.6          44       0.572      0.727   -0.155
0.7-0.8          15       0.750      1.000   -0.250
0.9-1.0          50       0.953      0.940   +0.013

calibration error: raw 0.0274 -> isotonic 0.0236


## The expected-value rule

$$EV(\text{release}) = (1-p)\,m\,A \;-\; p\,(A+f) \;-\; c_{review}$$

You earn *margin* on a recovered sale. You lose the *entire basket* plus
overhead on a bad one. That asymmetry is the whole ballgame, and it produces a
result that surprised me when it fell out of the code.

In [4]:
print('EV of releasing a Rs 1,00,000 order at various p_bad (margin 25%):\n')
for pb in (0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.40):
    print(f'  p_bad={pb:<6} EV = Rs {ev_release(pb, 100_000, cfg):>+10,.0f}')
print(f'\nbreak-even at p_bad = m/(1+m) = {ev_release_ceiling(cfg):.4f}')
print('\nThis is INDEPENDENT of order size. Contribution margin -- not risk')
print('appetite -- sets the ceiling on how much doubt a release can carry.')
for m_ in (0.15, 0.25, 0.40, 0.60):
    print(f'  margin {m_:.0%}  ->  can never release above p_bad {m_/(1+m_):.3f}')

EV of releasing a Rs 1,00,000 order at various p_bad (margin 25%):

  p_bad=0.0    EV = Rs    +25,000
  p_bad=0.05   EV = Rs    +18,712
  p_bad=0.1    EV = Rs    +12,425
  p_bad=0.15   EV = Rs     +6,138
  p_bad=0.2    EV = Rs       -150
  p_bad=0.25   EV = Rs     -6,438
  p_bad=0.4    EV = Rs    -25,300

break-even at p_bad = m/(1+m) = 0.2000

This is INDEPENDENT of order size. Contribution margin -- not risk
appetite -- sets the ceiling on how much doubt a release can carry.
  margin 15%  ->  can never release above p_bad 0.130
  margin 25%  ->  can never release above p_bad 0.200
  margin 40%  ->  can never release above p_bad 0.286
  margin 60%  ->  can never release above p_bad 0.375


This is a real difference from the sweep table in `IDEA.md` 7, which runs a raw
probability threshold out to 0.50. Under an expected-value policy at a 25%
margin, thresholds above 0.20 are unreachable -- the EV term forbids them long
before the risk cap does.

In [5]:
front = frontier(run(store, model, cfg, 'holdout'), vault, cfg, su)
print(f'{"cap":>7}{"released":>10}{"prec":>8}{"recall":>8}{"recovered":>14}{"admitted":>13}{"contribution":>15}')
print('-'*76)
for r in front:
    print(f'{r["cap"]:>7.3f}{r["released"]:>10}{r["precision"]:>8.3f}{r["recall"]:>8.3f}'
          f'{r["recovered"]:>14,.0f}{r["admitted"]:>13,.0f}{r["contribution"]:>15,.0f}')
print('\nnote where it stops moving.')

    cap  released    prec  recall     recovered     admitted   contribution
----------------------------------------------------------------------------
  0.005      1124   0.985   0.771    69,451,825    2,257,094     15,091,763
  0.010      1124   0.985   0.771    69,451,825    2,257,094     15,091,763
  0.020      1181   0.986   0.811    70,411,742    2,257,094     15,331,742
  0.030      1196   0.986   0.822    70,843,770    2,257,094     15,439,749
  0.050      1213   0.984   0.832    71,148,045    2,323,278     15,448,133
  0.075      1245   0.982   0.852    71,447,599    2,529,529     15,313,771
  0.100      1303   0.967   0.878    71,983,694    4,832,541     13,129,782
  0.125      1303   0.967   0.878    71,983,694    4,832,541     13,129,782
  0.150      1314   0.963   0.882    72,127,282    5,120,703     12,873,768
  0.175      1314   0.963   0.882    72,127,282    5,120,703     12,873,768
  0.200      1314   0.963   0.882    72,127,282    5,120,703     12,873,768

note where

## Four actions, not two

A system that only says yes or no is lying about the cases where it does not
know. There are four outcomes, and two of them are refusals.

In [6]:
class Case:
    def __init__(s, amt, orders=25, tenure=900):
        s.payment_id='pay_demo'; s.amount_inr=amt
        s.network={'network_orders_prior':orders,'network_tenure_days':tenure}

print('SAME p_bad, DIFFERENT amounts -> different actions:\n')
print(f'{"p_bad":>7}{"Rs 500":>12}{"Rs 50,000":>14}{"Rs 5,00,000":>14}')
print('-'*48)
for pb in (0.01, 0.10, 0.35, 0.60, 0.90):
    row = [decide(pb, Case(a), cfg).action.value for a in (500, 50_000, 500_000)]
    print(f'{pb:>7.2f}{row[0]:>12}{row[1]:>14}{row[2]:>14}')

print('\n\nTHIN FILE -- the abstention gate. Note it ignores p_bad entirely:\n')
for pb in (0.001, 0.05, 0.50, 0.99):
    d = decide(pb, Case(500_000, orders=1, tenure=120), cfg)
    print(f'  p_bad={pb:<7} -> {d.action.value:<10} {d.reasons[0]}')

SAME p_bad, DIFFERENT amounts -> different actions:

  p_bad      Rs 500     Rs 50,000   Rs 5,00,000
------------------------------------------------
   0.01    OVERTURN      OVERTURN      OVERTURN
   0.10      UPHOLD      OVERTURN      OVERTURN
   0.35      UPHOLD       STEP_UP       STEP_UP
   0.60      UPHOLD        UPHOLD        UPHOLD
   0.90      UPHOLD        UPHOLD        UPHOLD


THIN FILE -- the abstention gate. Note it ignores p_bad entirely:

  p_bad=0.001   -> STEP_UP    insufficient_evidence
  p_bad=0.05    -> STEP_UP    insufficient_evidence
  p_bad=0.5     -> STEP_UP    insufficient_evidence
  p_bad=0.99    -> UPHOLD     confidently_bad(p_bad=0.990>0.55)


That second table is the important one. The gate fires on **evidence quantity**,
never on model confidence. A gradient-boosted model will happily emit 0.001 on
a one-order file; that number is not knowledge, it is the prior wearing a
costume. The system refuses regardless.

## What the replay produces

In [7]:
ledger = run(store, model, PolicyConfig(cap=0.02), 'holdout')
out = grade(ledger, vault, su)
print('decision mix:', ledger.counts())
print(f'\nabstained on {out.abstention_rate:.1%} of cases  (step-up {ledger.counts()["STEP_UP"]}, escalate {ledger.counts()["ESCALATE"]})')
print(f'released {out.n_released} at {out.precision:.1%} precision')
print(f'recovered Rs {out.recovered_inr/1e7:.2f} cr, admitted Rs {out.fraud_admitted_inr/1e5:.2f} L')
print(f'net contribution Rs {out.net_contribution_inr/1e7:.2f} cr')
print('\nand one sample ledger record:\n')
print(ledger.records[7].to_json())

decision mix: {'OVERTURN': 855, 'UPHOLD': 460, 'STEP_UP': 451, 'ESCALATE': 9}

abstained on 25.9% of cases  (step-up 451, escalate 9)
released 1181 at 98.6% precision
recovered Rs 7.04 cr, admitted Rs 22.57 L
net contribution Rs 1.53 cr

and one sample ledger record:

{"action":"STEP_UP","amount_inr":254100.13,"block_reason":"amount_anomaly","created_at":1780289730,"day":0,"ev_release_inr":49053.4342,"evidence_sufficient":true,"merchant":"Aurum Jewels","network_orders_prior":43.0,"network_tenure_days":812.0,"p_bad":0.045454545,"payment_id":"pay_7A2VV7ZoBh5as9","reasons":["ambiguous","ev=+49,053","p_bad=0.045","amount_above_stepup_floor"],"seq":7}
